# 📗 Module 04 · Notebook 06 — Galeri Spesialis SLM

Di notebook 05 kita melatih **satu** spesialis (ekstraksi JSON). Di sini kita buktikan **resep
yang sama (QLoRA di Qwen3-0.6B) menggeneralisasi** ke banyak tugas dunia nyata:

1. **Klasifikasi / intent routing** — pesan pelanggan → intent
2. **Sentimen + aspect** — ulasan → `{aspek, sentimen}`
3. **Asisten domain** — Q&A toko (pengetahuan yang base TAK tahu)

+ **Benchmark ukuran**: 0.6B vs 1.7B vs 3B (memori & throughput inferensi).

Pola besarnya: *base lemah → fine-tune singkat → spesialis jago.* Sekali kuasai resepnya, kamu
bisa cetak SLM spesialis untuk tugas apa pun.

In [ ]:
!pip install -q "transformers>=4.53,<5" peft bitsandbytes accelerate datasets

In [ ]:
import torch, json, re, time, random
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SPECIALIST = "Qwen/Qwen3-0.6B"
GENERALIST = "HuggingFaceTB/SmolLM3-3B"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
def gpu_mem(tag=""):
    if torch.cuda.is_available():
        print(f"[{tag}] allocated={torch.cuda.memory_allocated()/1e9:.2f} GB")

## Resep yang Sama untuk Semua Tugas

Kita tulis **satu** fungsi `run_task(...)` yang: muat Qwen3-0.6B 4-bit → ukur base → QLoRA
fine-tune → ukur spesialis → bebaskan memori. Tiap tugas tinggal kasih: system prompt, data
latih (pasangan teks→jawaban), data uji, dan cara menilainya.

In [ ]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

results = []

def load_4bit(name):
    bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    tok=AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    mdl=AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb, device_map="auto")
    return mdl, tok

def chat_text(tok, sys, user, assistant=None):
    msgs=[{"role":"system","content":sys},{"role":"user","content":user}]
    if assistant is not None: msgs.append({"role":"assistant","content":assistant})
    add_gen = assistant is None
    try:
        return tok.apply_chat_template(msgs, add_generation_prompt=add_gen, tokenize=False, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, add_generation_prompt=add_gen, tokenize=False)

def gen(model, tok, sys, user, max_new_tokens=48):
    enc=tok(chat_text(tok, sys, user), return_tensors="pt").to(model.device)
    out=model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def parse_json(s):
    m=re.search(r"\{.*\}", s, re.S)
    if not m: return None
    try: return json.loads(m.group())
    except Exception: return None

def train_specialist(model, tok, pairs, sys, epochs=6):
    ds=Dataset.from_list([{"text": chat_text(tok, sys, u, a)} for u,a in pairs])
    ds=ds.map(lambda e: tok(e["text"], truncation=True, max_length=160), remove_columns=["text"])
    model=prepare_model_for_kbit_training(model)
    model=get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM", target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
    args=TrainingArguments(output_dir="sp", per_device_train_batch_size=8, gradient_accumulation_steps=1,
        num_train_epochs=epochs, learning_rate=2e-4, fp16=True, logging_steps=25,
        optim="paged_adamw_8bit", report_to="none", save_strategy="no")
    Trainer(model=model, args=args, train_dataset=ds,
            data_collator=DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)).train()
    model.config.use_cache=True; model.eval()
    return model

def run_task(name, sys, train_pairs, test_items, scorer, epochs=6):
    print(f"\n=== {name} ===")
    model, tok = load_4bit(SPECIALIST)
    base = scorer(model, tok, sys, test_items)
    model = train_specialist(model, tok, train_pairs, sys, epochs)
    spec = scorer(model, tok, sys, test_items)
    print(f"  base 0.6B = {base:.0%}   ->   SPESIALIS 0.6B = {spec:.0%}")
    del model, tok; torch.cuda.empty_cache()
    results.append({"task":name,"base":base,"spec":spec})

# --- penilai per jenis tugas ---
def score_cls(model, tok, sys, items):       # items: (teks, label)
    return sum(lab.lower() in gen(model,tok,sys,t,max_new_tokens=12).lower() for t,lab in items)/len(items)

def score_json(model, tok, sys, items, keys): # items: (teks, dict gold)
    ok=0
    for t,g in items:
        obj=parse_json(gen(model,tok,sys,t))
        if obj and all(str(obj.get(k)).strip().lower()==str(g[k]).strip().lower() for k in keys): ok+=1
    return ok/len(items)

def score_contains(model, tok, sys, items):   # items: (pertanyaan, key fakta)
    return sum(key.lower() in gen(model,tok,sys,q,max_new_tokens=48).lower() for q,key in items)/len(items)

def bench_batch_tps(model, tok, prompts, n_tokens=32):
    texts=[chat_text(tok, "You are helpful.", p) for p in prompts]
    old=tok.padding_side; tok.padding_side="left"
    enc=tok(texts, return_tensors="pt", padding=True).to(model.device); tok.padding_side=old
    model.generate(**enc, max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0=time.time()
    model.generate(**enc, min_new_tokens=n_tokens, max_new_tokens=n_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    return len(prompts)*n_tokens/(time.time()-t0)

## Tugas 1 — Klasifikasi / Intent Routing

Pesan pelanggan → satu **intent** (`tanya_harga`, `lacak_pesanan`, `komplain`, `refund`,
`info_produk`). Use-case klasik customer service: arahkan pesan ke divisi yang tepat.

In [ ]:
random.seed(1)
PROD=["beras","minyak","gula","kopi","sabun","mie"]
INTENT_TPL={
 "tanya_harga":["Berapa harga {p}?","{p} harganya berapa ya?","Mau tanya harga {p} dong","Harga {p} berapaan?"],
 "lacak_pesanan":["Pesanan saya sampai mana?","Tolong cek status order saya","Kok barang belum datang?","Mau lacak paket saya"],
 "komplain":["Barang yang saya terima rusak","{p} nya cacat, kecewa","Pesanan tidak sesuai, mau komplain","Kualitas {p} jelek"],
 "refund":["Saya mau refund","Tolong kembalikan uang saya","Cara minta uang kembali gimana?","Mau ajukan pengembalian dana"],
 "info_produk":["{p} ada stok?","{p} ready ga?","Spesifikasi {p} apa?","{p} tersedia warna apa?"],
}
def make_cls(n):
    out=[]
    for _ in range(n):
        intent=random.choice(list(INTENT_TPL))
        out.append((random.choice(INTENT_TPL[intent]).format(p=random.choice(PROD)), intent))
    return out
cls_train, cls_test = make_cls(150), make_cls(40)
SYS_CLS="Klasifikasikan pesan pelanggan ke satu intent: tanya_harga, lacak_pesanan, komplain, refund, info_produk. Jawab HANYA nama intent."
print("contoh:", cls_train[0])

In [ ]:
run_task("Klasifikasi intent", SYS_CLS, cls_train, cls_test, score_cls)

## Tugas 2 — Sentimen + Aspect

Ulasan → `{"aspek":..., "sentimen":...}`. Aspek: pengiriman/kualitas/harga/pelayanan; sentimen:
positif/negatif. Lebih kaya dari sekadar "positif/negatif" — tahu **apa** yang dipuji/dikeluhkan.

In [ ]:
ASP_TPL={
 ("pengiriman","positif"):["Pengiriman cepat banget, mantap","Paket datang lebih cepat dari perkiraan","Kurirnya ramah, kirim kilat"],
 ("pengiriman","negatif"):["Pengiriman lama sekali","Paket telat berhari-hari","Kurir lambat, kecewa"],
 ("kualitas","positif"):["Barangnya berkualitas, awet","Kualitas produk bagus sekali","Bahannya premium, suka"],
 ("kualitas","negatif"):["Barang cepat rusak","Kualitas jelek, mengecewakan","Produknya gampang patah"],
 ("harga","positif"):["Harganya murah meriah","Worth it banget harganya","Murah untuk kualitas segini"],
 ("harga","negatif"):["Kemahalan untuk barang segini","Harga tidak sebanding","Mahal banget, nyesel"],
 ("pelayanan","positif"):["Pelayanannya ramah dan cepat","CS membantu sekali","Penjual responsif dan sopan"],
 ("pelayanan","negatif"):["CS lama balas, judes","Pelayanan buruk","Penjual tidak responsif"],
}
def make_sent(n):
    out=[]
    for _ in range(n):
        (asp,sen)=random.choice(list(ASP_TPL))
        out.append((random.choice(ASP_TPL[(asp,sen)]), {"aspek":asp,"sentimen":sen}))
    return out
sent_train, sent_test = make_sent(150), make_sent(40)
SYS_SENT='Tentukan aspek (pengiriman/kualitas/harga/pelayanan) dan sentimen (positif/negatif) dari ulasan. Jawab HANYA JSON dengan key aspek dan sentimen.'
print("contoh:", sent_train[0])

In [ ]:
sent_pairs=[(t, json.dumps(g, ensure_ascii=False)) for t,g in sent_train]
run_task("Sentimen/aspect", SYS_SENT, sent_pairs, sent_test,
         lambda m,tk,s,it: score_json(m,tk,s,it,["aspek","sentimen"]))

## Tugas 3 — Asisten Domain (Toko Sembako "Berkah")

Q&A pengetahuan toko yang **dibuat-buat** — base model TAK mungkin tahu. Setelah fine-tune,
spesialis hafal fakta toko. (Sama seperti notebook 03, tapi sebagai bagian dari galeri.)

In [ ]:
FAQ=[
 ("Berapa minimal belanja untuk gratis ongkir?","Minimal belanja Rp100.000 untuk gratis ongkir di Toko Berkah.","100.000"),
 ("Jam berapa toko buka?","Toko Berkah buka pukul 08.00 sampai 21.00 setiap hari.","08.00"),
 ("Apakah bisa bayar di tempat (COD)?","Ya, Toko Berkah melayani COD untuk area Jabodetabek.","COD"),
 ("Berapa lama pengiriman ke luar kota?","Pengiriman ke luar kota 2 sampai 4 hari kerja.","2 sampai 4"),
 ("Apakah ada garansi produk?","Toko Berkah memberi garansi tukar barang dalam 3 hari jika rusak.","3 hari"),
 ("Nomor kontak toko berapa?","Toko Berkah bisa dihubungi di WhatsApp 0812-3456-7890.","0812-3456-7890"),
 ("Metode pembayaran apa saja?","Toko Berkah menerima transfer bank, e-wallet, dan COD.","e-wallet"),
 ("Apakah melayani grosir?","Ya, pembelian grosir di Toko Berkah diskon mulai 50 pcs.","50 pcs"),
]
SYS_FAQ="Kamu asisten Toko Sembako Berkah. Jawab pertanyaan pelanggan singkat berdasarkan info toko."
faq_train=[(q,a) for q,a,k in FAQ]*4      # ulang agar cukup untuk dihafal
faq_test=[(q,k) for q,a,k in FAQ]
print(f"{len(FAQ)} fakta toko")

In [ ]:
run_task("Asisten domain", SYS_FAQ, faq_train, faq_test, score_contains, epochs=10)

## Benchmark Ukuran: 0.6B vs 1.7B vs 3B

Inferensi 4-bit pada ketiga ukuran — lihat trade-off **memori** & **throughput**. (Ingat dari
notebook 05: di T4, model kecil menang telak di memori; throughput serupa pada skala kecil.)

In [ ]:
print(f"{'model':16}{'VRAM':>10}{'tok/dtk (batch-16)':>20}")
sample=[t for t,_ in cls_test[:16]]
for name,lbl in [(SPECIALIST,"Qwen3-0.6B"),("Qwen/Qwen3-1.7B","Qwen3-1.7B"),(GENERALIST,"SmolLM3-3B")]:
    m,t=load_4bit(name)
    v=torch.cuda.memory_allocated()/1e9
    tps=bench_batch_tps(m,t,sample)
    print(f"{lbl:16}{v:>8.2f}GB{tps:>18.0f}")
    del m,t; torch.cuda.empty_cache()

In [ ]:
print("=== RINGKASAN GALERI (base 0.6B -> SPESIALIS 0.6B) ===")
print(f"{'tugas':24}{'base':>8}{'spesialis':>12}")
for r in results:
    print(f"{r['task']:24}{r['base']:>7.0%}{r['spec']:>11.0%}")
print("\nSatu resep QLoRA, banyak spesialis. Itulah kenapa SLM dirilis berbondong-bondong.")

## Ringkasan Modul 04

Modul 04 lengkap — siklus hidup LLM utuh, ditambah strategi SLM dunia nyata:

| Notebook | Tema |
|----------|------|
| 00 | BANGUN — Transformer dari nol |
| 01 | PAKAI — model instruct (Qwen2.5-3B) |
| 02 | PRODUKSIKAN — quantization, batching, streaming |
| 03 | ADAPT — fine-tuning QLoRA |
| 04 | UKUR — evaluasi (metrik, statistik, LLM-judge) |
| 05 | SLM spesialis — kecil mengalahkan besar |
| 06 | Galeri spesialis — satu resep, banyak tugas |

**Selanjutnya: Module 05 — RAG.**